In [56]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F

from skimage.transform import resize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

In [57]:
def load_nifti(path):
    return nib.load(path).get_fdata()

def preprocess(vol):
    vol = resize(vol, (16,16,16))
    vol = (vol - np.mean(vol)) / (np.std(vol) + 1e-5)
    return vol

In [58]:
def extract_features(vol):
    return np.concatenate([
        np.mean(vol, axis=(1,2)),
        np.mean(vol, axis=(0,2)),
        np.mean(vol, axis=(0,1)),
        [np.std(vol)]   # 🔥 add global variation
    ])

In [59]:
def load_dataset(paths):
    volumes, labels = [], []

    label_map = {
        "healthy": 0,
        "alzheimer_early": 1,
        "alzheimer_diagnosed": 2,
        "parkinson_early":3,
        "parkinson_diagnosed": 4
    }

    for label, folder in paths.items():
        for f in os.listdir(folder):
            if f.endswith(".nii") or f.endswith(".nii.gz"):
                v = preprocess(load_nifti(os.path.join(folder, f)))
                volumes.append(v)
                labels.append(label_map[label])

    return volumes, np.array(labels)

In [60]:
paths = {
    "healthy": "alzimers/cn",
    "alzheimer_early": "alzimers/mci",
    "alzheimer_diagnosed": "alzimers/ad",
    "parkinson_early": "parkinsons/detect",
    "parkinson_diagnosed": "parkinsons/disease"
}

volumes, labels = load_dataset(paths)

In [61]:
train_idx, test_idx = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [62]:
train_mask = torch.zeros(len(labels), dtype=torch.bool)
test_mask = torch.zeros(len(labels), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [63]:
features = np.array([extract_features(v) for v in volumes])

scaler = StandardScaler()
features = scaler.fit_transform(features)

features = torch.tensor(features, dtype=torch.float)
labels = torch.tensor(labels, dtype=torch.long)
# 🔥 Oversampling to strengthen disease learning
X_list = features.tolist()
y_list = labels.tolist()

for i in range(len(y_list)):
    if y_list[i] == 0:   # duplicate disease classes (1 & 2)
        X_list.append(X_list[i])
        y_list.append(y_list[i])

# convert back
features = torch.tensor(X_list, dtype=torch.float)
labels = torch.tensor(y_list, dtype=torch.long)

In [64]:
# rebuild graph AFTER final features & labels
sim = cosine_similarity(features.numpy())

edge_index = []
edge_weight = []

for i in range(len(features)):
    for j in range(len(features)):
        if i != j and sim[i][j] > 0.6:
            edge_index.append([i, j])
            edge_weight.append(sim[i][j])

edge_index = torch.tensor(edge_index).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

graph_data = Data(
    x=features,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=labels
)

In [65]:
train_idx, test_idx = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_mask = torch.zeros(len(labels), dtype=torch.bool)
test_mask = torch.zeros(len(labels), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [66]:
graph_data = Data(
    x=features,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=labels
)

In [67]:
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(49, 32)
        self.conv2 = GCNConv(32, 16)
        self.conv3 = GCNConv(16, 8)
        self.fc = torch.nn.Linear(8, 5)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index, data.edge_weight))
        x = F.dropout(x, p=0.4, training=self.training)

        x = F.relu(self.conv2(x, data.edge_index, data.edge_weight))
        x = F.dropout(x, p=0.3, training=self.training)

        x = F.relu(self.conv3(x, data.edge_index, data.edge_weight))
        x = self.fc(x)

        return F.log_softmax(x, dim=1)

In [68]:
model = GNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
weights = torch.tensor([3.0, 1.0, 1.0])  # boost class 0
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    out = model(graph_data)
    loss = F.nll_loss(out[train_mask], labels[train_mask], weight=weights)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

RuntimeError: weight tensor should be defined either for all 5 classes or no classes but got weight tensor of shape: [3]

In [ ]:
model.eval()
out = model(graph_data)

pred = out[test_mask].argmax(dim=1)
acc = (pred == labels[test_mask]).sum().item() / test_mask.sum().item()

print("Test Accuracy:", acc)

Test Accuracy: 0.5666666666666667


In [ ]:
model.eval()
out = model(graph_data)

pred = out[test_mask].argmax(dim=1)
acc = (pred == y[test_mask]).sum().item() / test_mask.sum().item()

print("Test Accuracy:", acc)

Test Accuracy: 0.5555555555555556
